# 中金 · 股债双杀预警复现

复现《巧识尾部相关性，规避组合系统性风险》（中金量化及ESG, 2021-02-05）。

## 原文口径

| 项 | 原文 |
|---|---|
| 股票 | 沪深300 |
| 债券 | 中债综合财富指数 |
| 窗口 | 60 个交易日，每日滚动 |
| 回测区间 | 2006–2021 |
| 预警次数 | 15 年共 5 次（2008/2013/2015/2016/2020） |

**触发条件**（三条同时满足）：λ_D > 0、λ_U = 0、股或债至少一方 DIF < 0
**解除条件**（两条同时满足）：λ_D = 0、债券 DIF > 0

## 复现的定位

目标不是复现出与研报一致的收益曲线。原文样本量很小（15 年仅 5 次预警），
对不上是常态。目标是回答两个可证伪的问题：

1. **λ_D>0 的出现频率是否显著超过噪声基准？**（否则测出的是噪声）
2. **情景③（λ_D>0 且 λ_U=0）的股债次日收益是否双负？**（这是原文的核心证据）

## 0 · 环境与数据

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tailrisk_core as tc

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.unicode_minus"] = False
for f in ["Arial Unicode MS", "SimHei", "PingFang SC"]:
    try:
        plt.rcParams["font.sans-serif"] = [f]; break
    except Exception: pass

### 债券标的的选择

中债综合财富指数在聚宽和同花顺都拿不到长历史（同花顺 trial 账号仅滚动 5 年）。
下面三个选项并存，**改 `BOND` 一行即可切换**，其余代码不用动：

| 选项 | 区间 | 券种 | 说明 |
|---|---|---|---|
| `上证国债指数` | 2005–今 | 纯国债 | 原文的趋势条件与回测基准本就用国债指数 |
| `上证企业债指数` | 2005–2020 | 企业债 | 含信用成分，ties 最干净，覆盖全部 5 次预警 |
| `中债综合财富指数` | 2021–今 | 综合 | 口径最贴原文，但只有 5 年 |

In [ ]:
import os
import jqdatasdk as jq


def jq_login():
    """
    聚宽登录，三级回退：环境变量 → credentials.py → 交互输入。

    为什么不直接用 os.environ：终端里 export 的变量只有从该终端启动的进程
    才继承得到。从 Dock/Spotlight 点开的 VS Code 拿不到，其内核也拿不到。
    Jupyter 若从终端启动则没问题。三级回退让哪种启动方式都能跑。

    ⚠ credentials.py 记得加进 .gitignore，别让密码进版本库或跟着压缩包发出去。
    """
    u, p = os.environ.get("JQ_USERNAME"), os.environ.get("JQ_PASSWORD")
    if u and p:
        src = "环境变量"
    else:
        try:
            import credentials
            u, p = credentials.JQ_USERNAME, credentials.JQ_PASSWORD
            src = "credentials.py"
        except (ImportError, AttributeError):
            from getpass import getpass
            u, p = input("聚宽账号: "), getpass("密码: ")
            src = "交互输入"
    jq.auth(u, p)
    print(f"聚宽登录成功（凭据来自{src}）")
    return jq


jq_login()

START, END = "2005-01-01", "2026-08-17"
EQUITY = "沪深300"
BOND   = "上证国债指数"     # ← 换标的只改这里

JQ_CODES = {"沪深300": "000300.XSHG", "上证国债指数": "000012.XSHG",
            "上证企业债指数": "000013.XSHG"}

px = {}
for name in [EQUITY, BOND]:
    if name in JQ_CODES:
        px[name] = jq.get_price(JQ_CODES[name], start_date=START,
                                end_date=END, fields=["close"])["close"].dropna()
    else:
        ths = tc.parse_ifind_export("data/ifind_export.txt")
        col = [c_ for c_ in ths.columns if name.split("(")[0] in c_][0]
        px[name] = ths[col].dropna()

prices = pd.DataFrame(px).dropna()
print(f"共同交易日 {len(prices)} 天：{prices.index.min():%Y-%m-%d} ~ {prices.index.max():%Y-%m-%d}")
prices.head()

In [ ]:
quality = tc.data_quality(prices)
quality

In [ ]:
returns = prices.pct_change().dropna()
returns.describe().T[["count", "mean", "std", "min", "max"]]

## 1 · 概率积分变换

主口径用 `empirical`（最贴近原文，原文未提任何滤波）。
`garch` 作为对照版本一起算，最后并排出结果。

In [ ]:
pit = pd.DataFrame({
    f"{EQUITY}_emp": tc.pit_empirical(returns[EQUITY]),
    f"{BOND}_emp":   tc.pit_empirical(returns[BOND]),
})
pit.describe().T[["count", "mean", "std", "min", "max"]]

## 2 · 滚动尾部相关系数

60 日窗口，每日重估。Clayton 与 Gumbel 各自独立拟合。

⚠ 这一格较慢（每天两次 MLE，5000 个交易日约 10–30 分钟）。
先用 `STEP=5` 看方向，确认后再改回 1。

In [ ]:
STEP = 5     # ← 正式跑改成 1

roll = tc.rolling_tail_dependence(pit[f"{EQUITY}_emp"], pit[f"{BOND}_emp"],
                                  window=60, step=STEP)
print(f"{len(roll)} 个重估点")
roll.head()

### 2.1 与噪声基准对照 ← 最关键的一步

In [ ]:
base = tc.noise_baseline(window=60, n_rep=300, rho=0.0, seed=1)

ld_pos = roll.lambda_D > tc.LAMBDA_ZERO_TOL
lu_pos = roll.lambda_U > tc.LAMBDA_ZERO_TOL
cicc_cond = ld_pos & ~lu_pos

compare = pd.DataFrame({
    "实测": [ld_pos.mean(), lu_pos.mean(), cicc_cond.mean()],
    "纯独立数据基准": [base["lambda_D>0占比"], base["lambda_U>0占比"],
                       base["中金条件命中率"]],
}, index=["λ_D>0 占比", "λ_U>0 占比", "中金条件(λ_D>0且λ_U=0)"])
compare["倍数"] = compare["实测"] / compare["纯独立数据基准"]
compare

**判读**：

- `倍数` 接近 1 → 测出的与噪声不可区分，**信号未产生**，后面所有结果都没有意义
- `倍数` 明显大于 1（建议 ≥ 2.5）→ 信号存在，继续往下看

首轮用国债ETF跑出来的倍数是 1.0（实测 2.0% vs 基准 2.0%），据此判定信号未产生。

In [ ]:
ct = pd.crosstab(ld_pos.map({True:"λ_D>0", False:"λ_D=0"}),
                 lu_pos.map({True:"λ_U>0", False:"λ_U=0"}))
print("λ_D × λ_U 联合分布（天数）:"); display(ct)
print("占比 (%):"); display((ct / ct.values.sum() * 100).round(1))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(roll.index, roll.lambda_D, lw=.7, color="firebrick")
axes[0].axhline(0, color="k", lw=.5); axes[0].set_ylabel("λ_D")
axes[0].set_title(f"滚动尾部相关系数  {EQUITY} × {BOND}")
axes[1].plot(roll.index, roll.lambda_U, lw=.7, color="steelblue")
axes[1].axhline(0, color="k", lw=.5); axes[1].set_ylabel("λ_U")
for ax in axes: ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3 · 趋势条件

标准 DIF = MA(12) − MA(26)，对股债分别计算。

In [ ]:
trend = pd.DataFrame({
    f"DIF_{EQUITY}": tc.dif_trend(prices[EQUITY]),
    f"DIF_{BOND}":   tc.dif_trend(prices[BOND]),
}).reindex(roll.index)

trend["股下跌"] = trend[f"DIF_{EQUITY}"] < 0
trend["债下跌"] = trend[f"DIF_{BOND}"] < 0
trend["至少一方下跌"] = trend["股下跌"] | trend["债下跌"]
trend.tail()

In [ ]:
trend[["股下跌", "债下跌", "至少一方下跌"]].mean().to_frame("占比")

## 4 · 预警状态机

触发：λ_D>0 且 λ_U=0 且（股或债 DIF<0）
解除：λ_D=0 且 债券 DIF>0

In [ ]:
sig = pd.DataFrame(index=roll.index)
sig["lambda_D"] = roll.lambda_D
sig["lambda_U"] = roll.lambda_U
sig["触发条件"] = (roll.lambda_D > tc.LAMBDA_ZERO_TOL) & \
                  (roll.lambda_U <= tc.LAMBDA_ZERO_TOL) & \
                  trend["至少一方下跌"]
sig["解除条件"] = (roll.lambda_D <= tc.LAMBDA_ZERO_TOL) & (trend[f"DIF_{BOND}"] > 0)

# 状态机：触发后保持预警，直到解除条件满足
state, states = False, []
for t, r in sig.iterrows():
    if not state and r["触发条件"]:
        state = True
    elif state and r["解除条件"]:
        state = False
    states.append(state)
sig["预警中"] = states

print(f"预警天数 {sig['预警中'].sum()} / {len(sig)} ({sig['预警中'].mean():.2%})")
sig.head()

In [ ]:
# 切分成事件区间
blocks = (sig["预警中"] != sig["预警中"].shift()).cumsum()
episodes = [(g.index.min(), g.index.max())
            for _, g in sig[sig["预警中"]].groupby(blocks[sig["预警中"]])]

ep_df = pd.DataFrame(episodes, columns=["起", "止"])
ep_df["持续交易日"] = [len(sig.loc[s:e]) for s, e in episodes]
print(f"共 {len(ep_df)} 次预警事件")
ep_df

### 4.1 与原文的 5 次预警核对

In [ ]:
EVENTS = {
    "2008金融危机":  ("2008-01-01", "2008-12-31"),
    "2013钱荒":      ("2013-06-01", "2013-12-31"),
    "2015股灾":      ("2015-06-01", "2015-09-30"),
    "2016末债灾":    ("2016-11-01", "2017-01-31"),
    "2020疫情":      ("2020-02-01", "2020-04-30"),
}

rows = []
for name, (s, e) in EVENTS.items():
    seg = sig.loc[s:e, "预警中"] if len(sig.loc[s:e]) else pd.Series(dtype=bool)
    rows.append({"事件": name, "区间": f"{s}~{e}",
                 "样本覆盖": len(seg) > 0,
                 "预警天数": int(seg.sum()) if len(seg) else 0,
                 "是否捕捉": bool(seg.any()) if len(seg) else None})
pd.DataFrame(rows)

## 5 · 情景对照表 ← 原文的核心证据

统计六种情景下**股债次日收益的均值**。原文披露：

| 情景 | 股票 | 债券 |
|---|---|---|
| ①历史所有时间 | +0.048% | +0.016% |
| ③λ_D≠0 且 λ_U=0 | **−0.123%** | **−0.026%** |

关键在情景③：**股债应同时为负**，而其余情景为正。

In [ ]:
fwd_e = returns[EQUITY].shift(-1).reindex(roll.index)
fwd_b = returns[BOND].shift(-1).reindex(roll.index)

win = 60
pear = returns[EQUITY].rolling(win).corr(returns[BOND]).reindex(roll.index)

roll_rank_corr = pd.Series(index=roll.index, dtype=float)
roll_kend = pd.Series(index=roll.index, dtype=float)
re_, rb_ = returns[EQUITY], returns[BOND]
for t in roll.index:
    seg_e = re_.loc[:t].tail(win); seg_b = rb_.loc[:t].tail(win)
    if len(seg_e) < win: continue
    roll_rank_corr[t] = seg_e.corr(seg_b, method="spearman")
    roll_kend[t] = seg_e.corr(seg_b, method="kendall")

scenarios = {
    "①历史所有时间":        pd.Series(True, index=roll.index),
    "②λ_D≠0":              ld_pos,
    "③λ_D≠0且λ_U=0":       cicc_cond,
    "④Pearson>0":          pear > 0,
    "⑤Spearman>0":         roll_rank_corr > 0,
    "⑥Kendall>0":          roll_kend > 0,
}

rows = []
for name, mask in scenarios.items():
    m = mask.fillna(False)
    rows.append({"情景": name, "样本天数": int(m.sum()),
                 "股票次日均收益": fwd_e[m].mean(),
                 "债券次日均收益": fwd_b[m].mean()})
scen = pd.DataFrame(rows)
scen["股债双负"] = (scen["股票次日均收益"] < 0) & (scen["债券次日均收益"] < 0)
scen.style.format({"股票次日均收益": "{:.4%}", "债券次日均收益": "{:.4%}"})

**验收**：情景③的「股债双负」为 True，且样本天数足够（>100）
——同时满足才谈得上复现了原文的核心证据。

样本天数太少时那个均值没有统计意义，不要据此下结论。

## 6 · 回测

基准：恒定 50% 股 + 50% 债
策略：预警触发次日降低股债敞口（转现金），解除次日恢复

In [ ]:
bt = pd.DataFrame(index=roll.index)
bt["r_e"] = returns[EQUITY].reindex(roll.index)
bt["r_b"] = returns[BOND].reindex(roll.index)
bt["预警中"] = sig["预警中"]

bt["基准"] = 0.5 * bt.r_e + 0.5 * bt.r_b
# 预警次日生效：用 shift(1)，避免用当天信号交易当天（前视偏差）
in_warn = bt["预警中"].shift(1).fillna(False)
bt["策略"] = np.where(in_warn, 0.0, bt["基准"])

nav = (1 + bt[["基准", "策略"]].fillna(0)).cumprod()

def stats(r):
    r = r.dropna()
    n = len(r); yrs = n / 244
    ann = (1 + r).prod() ** (1 / yrs) - 1
    vol = r.std() * np.sqrt(244)
    navc = (1 + r).cumprod()
    mdd = (navc / navc.cummax() - 1).min()
    return {"年化收益": ann, "年化波动": vol, "收益波动比": ann / vol if vol else np.nan,
            "最大回撤": mdd, "卡玛比率": ann / abs(mdd) if mdd else np.nan}

pd.DataFrame({k: stats(bt[k]) for k in ["基准", "策略"]}).T

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
nav.plot(ax=ax, lw=1.2)
for s, e in episodes:
    ax.axvspan(s, e, color="red", alpha=.12)
ax.set_title("净值对比（红色阴影=预警期）"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7 · 结论清单

复现是否成立，看这三条是否**同时**满足：

1. λ_D>0 占比显著超过噪声基准（第 2.1 节，倍数 ≥ 2.5）
2. 情景③股债双负，且样本天数 > 100（第 5 节）
3. 2013 钱荒、2016 债灾至少捕捉一次（第 4.1 节）

**任一不满足时的排查顺序**（不要直接改参数）：

1. 先看 λ_D>0 占比是否仍等于噪声基准 → 是则数据问题，换债券标的
2. 再看 λ_U 判零是否恒不成立 → 是则用 `ll_diff` 备选判定，两版本并排出
3. 最后才动边缘分布口径（换 `pit_garch` / `pit_vol_scaled` 对照）

**明确禁止**：调参数直到情景③变负。那是在制造想要的结论。